# Assignment 1

**Set Up**: Run code below to import `pandas`. 

In [1]:
# set up
import pandas as pd
import numpy as np

## 1. DataFrame formatting

In [ ]:
transaction = pd.read_csv("../data/data.zip")
transaction

FileNotFoundError: [Errno 2] No such file or directory: '../data/transaction_data.zip'

In [ ]:
meta = pd.read_csv("../data/meta_data.csv")
meta

,feature,description
0,sku_number,Eight character stock keeping unit number
1,inventory_type,Types of inventory including finished good - w...
2,stocking_type,Stockign type policy including make to stock -...
3,lead_time,Total lead time in days
4,unit_price,Unit price US dollar value
5,manufacturing_site,Manufacturing site ID starting with country co...
6,division_code,Four character business division ID
7,transaction_date,Date of Transaction in Year-Month-Day format
8,order_quantity,Order quantity - rounded up for convenience


In [ ]:
# renames the column labels for transaction data
transaction.columns = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price", "manufacturing_site", "division_code", "transaction_date", "order_quantity"]   #from 2_data_cleaning

## 2. Data Inspection

In [ ]:
transaction.describe() #from 2_data_cleaning

,lead_time,unit_price,order_quantity
count,784639.000000,784639.000000,784639.000000
mean,25.779881,798.642045,79.857114
std,10.990559,375.007339,41.954441
min,14.000000,-1008.385458,-120.000000
25%,28.000000,983.512015,63.000000
50%,28.000000,995.890776,93.000000
75%,28.000000,1004.855517,109.000000
max,2800.000000,1028.834974,189.000000


The `transaction` data contain three quanitative features; `lead_time`, `unit_price`, and `order_quanity`. 

Over all transactions, the average lead time is ~25.78 days and the median is 28 days. The lead time ranges from 14 days to 2,800 days. We will address the 10 occurances of a 2800 day lead time and correct them. 

The average unit price is \$798.64 and a median of \$995.89. Over all transactions, the unit price ranges from \$-1,008.39 and \$1,028.83. We will correct it and assume it was a data entry error because it does not make sense for a specific sku to have a negative unit price. 

The average order quanitity is 79.86 units and a median of 93 units. There is a range of -120 to 189 units over all transaction data. The negative order quanity is concerning, since it is safety stock we will be more cautious and assume it was meant to be positive values. 

In [ ]:
transaction.info()

<class 'pandas.DataFrame'>
RangeIndex: 784639 entries, 0 to 784638
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   sku_number          784469 non-null  str    
 1   inventory_type      775250 non-null  str    
 2   stocking_type       763084 non-null  str    
 3   lead_time           784639 non-null  int64  
 4   unit_price          784639 non-null  float64
 5   manufacturing_site  632214 non-null  str    
 6   division_code       729751 non-null  str    
 7   transaction_date    784639 non-null  str    
 8   order_quantity      784639 non-null  int64  
dtypes: float64(1), int64(2), str(6)
memory usage: 53.9 MB


The data types look ok except the `transaction_date` can be converted to a date time.

### Fix `transaction_date`:

In [ ]:
transaction["transaction_date"] = pd.to_datetime(transaction["transaction_date"], format='%Y-%m-%d') # from 2_data_cleaning
transaction.info()

<class 'pandas.DataFrame'>
RangeIndex: 784639 entries, 0 to 784638
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   sku_number          784469 non-null  str           
 1   inventory_type      775250 non-null  str           
 2   stocking_type       763084 non-null  str           
 3   lead_time           784639 non-null  int64         
 4   unit_price          784639 non-null  float64       
 5   manufacturing_site  632214 non-null  str           
 6   division_code       729751 non-null  str           
 7   transaction_date    784639 non-null  datetime64[us]
 8   order_quantity      784639 non-null  int64         
dtypes: datetime64[us](1), float64(1), int64(2), str(5)
memory usage: 53.9 MB


### Fix `lead_time`:

In [ ]:
transaction['lead_time'].value_counts()

lead_time
28      627914
14       96101
21       60614
2800        10
Name: count, dtype: int64

2800 days of lead time is an outlier and most likely a data entry error. We will review the entries containing 2800 to confirm they are otherwise normal in the other columns of data across those rows. 

In [ ]:
transaction[transaction['lead_time'] > 28]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
41007,3D05F3A5,FG,MTO,2800,1019.322038,NaN,3E44,2023-02-27,106
140935,6ED1D532,FG,MTS,2800,995.694539,JP-41,5E37,2023-07-16,99
210484,98A7A592,FG,MTS,2800,995.516935,NaN,219B,2023-10-21,101
245144,B366DFB0,RM,MTS,2800,1000.066895,US-E8,219B,2023-12-09,118
340984,0FF91D18,FG,MTS,2800,994.567451,CA-66,219B,2024-04-21,130
349417,FA47DC02,FG,MTO,2800,1007.789711,JP-41,6E75,2024-05-03,114
422092,B62739F8,WIP,MTS,2800,1006.039081,JP-41,0377,2024-08-12,102
428323,B66C227C,RM,MTO,2800,1000.117336,US-B4,D66F,2024-08-21,83
680839,BC01EE00,FG,MTS,2800,987.388003,JP-41,219B,2025-08-08,99
745116,913D99A7,FG,MTS,2800,989.254537,US-B4,219B,2025-11-06,75


In [ ]:
transaction[transaction['lead_time'] == 2800]
transaction.loc[
    transaction['lead_time'] == 2800,
    'lead_time'] = 28

After reviewing the rows and not seeing any other errors we can convert the 2800 to a 28 day lead time. A 28 day lead time seems very reasonable to assume since the majority of the sku's have a 28 day lead time, and average lead time is ~26 days. 

In [ ]:
#Check the lead time has been fixed
transaction['lead_time'].value_counts()

lead_time
28    627924
14     96101
21     60614
Name: count, dtype: int64

### Fix `unit_price`:

In [ ]:
transaction[transaction['unit_price'] < 0]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
22190,F6AEA3E6,FG,MTO,28,-990.170311,CA-B0,15DF,2023-02-01,99
100790,0D405CB4,FG,MTO,14,-47.591490,NaN,7454,2023-05-21,24
109720,522ACD1B,FG,MTO,28,-979.026922,US-B4,5E37,2023-06-03,142
118496,5FFD6CCA,FG,MTS,28,-1002.300571,US-B4,BBC9,2023-06-15,99
236942,2D035681,RM,MTO,21,-201.212700,CA-DC,D66F,2023-11-27,43
242639,DB51AF01,WIP,MTS,21,-197.813774,JP-41,69FC,2023-12-05,57
566269,9ACCB6FD,RM,MTO,14,-55.849836,CA-66,8399,2025-03-01,14
584799,6AC708C5,FG,MTS,28,-1003.846519,CA-66,5E37,2025-03-27,78
615119,A7381ECF,FG,MTO,28,-990.719583,US-B4,D0AA,2025-05-09,86
659480,AC300778,FG,MTS,28,-1008.385458,JP-41,A57B,2025-07-09,99


A negative unit price with no negative order quantity should not occur.
We can assume they are suppose to be positive values.

In [ ]:
transaction['unit_price'] = transaction['unit_price'].abs()

In [ ]:
transaction[transaction['unit_price'] < 0]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity


### Fix `order_quantity`:

In [ ]:
transaction[transaction['order_quantity'] < 0]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
46,F79848D2,RM,MTS,14,53.835146,CA-66,5E37,2023-01-01,-6
53,5BAADDE7,FG,MTS,14,52.561910,CA-66,3E44,2023-01-01,-4
57,0DC49275,FG,MTS,21,200.509633,JP-41,58D1,2023-01-01,-7
95,225E8CD2,FG,MTO,14,46.255435,JP-8F,6E75,2023-01-01,-9
115,8D0AD087,FG,MTO,14,38.236338,US-B4,BC95,2023-01-01,-1
...,...,...,...,...,...,...,...,...,...
784569,FE564816,FG,MTS,28,109.330222,NaN,6E75,2025-12-30,-9
784576,FCD26C09,RM,MTS,14,51.573804,US-95,5E37,2025-12-30,-15
784580,B4A45673,FG,MTO,21,196.110783,JP-41,BBC9,2025-12-30,-24
784612,B52AD9CE,FG,MTS,28,99.130877,JP-41,8399,2025-12-30,-3


Over 40,000 rows have a negative order quantity. Order quantity is needed to determine safety stock so we don't want to guess with these values. We will turn all these values positive and assume it was a data entry error.

In [ ]:
transaction['order_quantity'] = transaction['order_quantity'].abs()

In [ ]:
transaction[transaction['order_quantity'] < 0]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity


## 3. NaN Values

In [ ]:
transaction.isna().sum()

sku_number               170
inventory_type          9389
stocking_type          21555
lead_time                  0
unit_price                 0
manufacturing_site    152425
division_code          54888
transaction_date           0
order_quantity             0
dtype: int64

The only columns with no missing values is `lead_time`, `transaction_date`, `unit_price`, and `order_quantity`. All else needs to be addressed.

### Fix `sku_number`:

In [ ]:
transaction[transaction['sku_number'].isna()]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
411,NaN,WIP,MTO,14,61.914680,NaN,8399,2023-01-01,12
7204,NaN,FG,MTO,14,52.793522,NaN,15DF,2023-01-11,0
14824,NaN,WIP,MTO,14,61.914680,JP-41,15DF,2023-01-21,11
15686,NaN,FG,MTO,14,52.793522,NaN,17B8,2023-01-22,6
18160,NaN,FG,MTO,14,52.793522,CA-2B,8399,2023-01-26,9
...,...,...,...,...,...,...,...,...,...
750133,NaN,FG,MTO,14,52.793522,NaN,0377,2025-11-12,7
755056,NaN,WIP,MTO,14,61.914680,NaN,8399,2025-11-20,22
756730,NaN,WIP,MTO,14,61.914680,CA-66,5E37,2025-11-22,27
772620,NaN,FG,MTO,14,52.793522,CA-66,23F0,2025-12-14,13


There are 170 rows of data with missing SKUs. They can not be recovered and we have no way of checking what they could be. SKU is needed so we know what we are ordering when it comes to safety stock calculations, therefore we should remove those rows.

In [ ]:
transaction = transaction.dropna(subset=['sku_number'])
transaction['sku_number'].isna().sum()

np.int64(0)

Removed all rows in the `sku_number` column with `NaN`.

### Fix `inventory_type`, `stocking_type,` `manufacturing_site`, and `division_code`

There is almost 10,000 rows of missing data throughout the categorical variables. None of these are needed to determine safety stock and are not integer values. We can replace these with "missing"

In [ ]:
# changes 'inventory_type', 'stocking_type', 'manufacturing_site', 'division_code' n/a's to missing
Transaction_ValuesMissing = ['inventory_type', 'stocking_type', 'manufacturing_site', 'division_code'] 
transaction[Transaction_ValuesMissing] = transaction[Transaction_ValuesMissing].fillna("missing")
transaction

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
0,DB45C19C,FG,MTO,28,992.860458,JP-41,missing,2023-01-01,130
1,83992E02,missing,MTS,21,202.017834,CA-66,ACA9,2023-01-01,39
2,75C364B0,FG,MTS,28,986.009314,JP-41,23F0,2023-01-01,111
3,2C9772C9,missing,MTS,28,1007.216590,CA-66,9B75,2023-01-01,89
4,ACCCE163,WIP,MTS,28,986.998503,CA-66,6E75,2023-01-01,80
...,...,...,...,...,...,...,...,...,...
784634,F8B91C1B,WIP,MTS,28,998.021118,missing,23F0,2025-12-30,111
784635,6664F459,FG,MTS,28,99.200335,CA-66,D66F,2025-12-30,1
784636,3DE1D314,FG,MTS,28,1008.523300,missing,81FE,2025-12-30,115
784637,DB131462,FG,MTS,28,1008.340512,CA-2B,6AF1,2025-12-30,122


- `inventory_type`: This is the status within the process; work in progress, finished good, or raw material. We changed it to `"missing"` because we want to preserve those rows or data, but the inventory type is not need for the safety stock calculation. 
- `stocking_type`: This tells us how the inventory is managed; make to stock, make to order, or assemble to order. There is a similar apprach as above, we want to preserve the rows of data, and the stocking type is not essential for safety stock calculation. 
- `manufacturing_site`: This is an ID starting with country ID. We can't know where it came from if the value is missing, but we still want the other information for the remaining feauture so we will mark as `"missing"`. 
- `division_code`: Similar to the problem above, it is an unknown ID and it is not essential for our future calculations, so we will mark as missing to preserve the remaining data. 

In [ ]:
transaction.isna().sum()
# selected columns have been succesfully replaces with "missing"

sku_number            0
inventory_type        0
stocking_type         0
lead_time             0
unit_price            0
manufacturing_site    0
division_code         0
transaction_date      0
order_quantity        0
dtype: int64

## 4. Extracting the Country Values


In [ ]:
# Meta data tells us the country codes are us, pl, ca, and jp
country_values = {"US":"United States", "PL":"Poland", "CA":"Canada", "JP":"Japan"}

transaction["country"]= "missing"

# The country code is located in the first two slots of the manufacturing site data. We can use str.startwith() to extract it from manufacture site. 
transaction.loc[transaction["manufacturing_site"].str.startswith("US"), "country"]=country_values["US"]
transaction.loc[transaction["manufacturing_site"].str.startswith("PL"), "country"]=country_values["PL"]
transaction.loc[transaction["manufacturing_site"].str.startswith("CA"), "country"]=country_values["CA"]
transaction.loc[transaction["manufacturing_site"].str.startswith("JP"), "country"]=country_values["JP"]

transaction

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
0,DB45C19C,FG,MTO,28,992.860458,JP-41,missing,2023-01-01,130,Japan
1,83992E02,missing,MTS,21,202.017834,CA-66,ACA9,2023-01-01,39,Canada
2,75C364B0,FG,MTS,28,986.009314,JP-41,23F0,2023-01-01,111,Japan
3,2C9772C9,missing,MTS,28,1007.216590,CA-66,9B75,2023-01-01,89,Canada
4,ACCCE163,WIP,MTS,28,986.998503,CA-66,6E75,2023-01-01,80,Canada
...,...,...,...,...,...,...,...,...,...,...
784634,F8B91C1B,WIP,MTS,28,998.021118,missing,23F0,2025-12-30,111,missing
784635,6664F459,FG,MTS,28,99.200335,CA-66,D66F,2025-12-30,1,Canada
784636,3DE1D314,FG,MTS,28,1008.523300,missing,81FE,2025-12-30,115,missing
784637,DB131462,FG,MTS,28,1008.340512,CA-2B,6AF1,2025-12-30,122,Canada


## 5. Helpful Dimensional Information

We will begin with making a subset for rows with `iventory_type` labeled as a finished goods (`FG`). 

In [ ]:
transaction_FG_MTS = transaction.query("inventory_type == 'FG' and stocking_type == 'MTS'")
transaction_FG_MTS

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
2,75C364B0,FG,MTS,28,986.009314,JP-41,23F0,2023-01-01,111,Japan
14,D89A6000,FG,MTS,28,988.743492,US-B4,219B,2023-01-01,75,United States
15,53072532,FG,MTS,14,59.582062,JP-41,81FE,2023-01-01,9,Japan
18,99693AF2,FG,MTS,28,990.361032,CA-66,B21F,2023-01-01,78,Canada
21,075F66DC,FG,MTS,28,1001.812924,JP-41,15DF,2023-01-01,84,Japan
...,...,...,...,...,...,...,...,...,...,...
784626,92E72E5D,FG,MTS,28,1003.277928,CA-66,15DF,2025-12-30,108,Canada
784635,6664F459,FG,MTS,28,99.200335,CA-66,D66F,2025-12-30,1,Canada
784636,3DE1D314,FG,MTS,28,1008.523300,missing,81FE,2025-12-30,115,missing
784637,DB131462,FG,MTS,28,1008.340512,CA-2B,6AF1,2025-12-30,122,Canada


### How many unique SKUs are we working with?

In [ ]:
transaction_FG_MTS["sku_number"].drop_duplicates().count()

np.int64(368)

In [ ]:
transaction_FG_MTS["sku_number"].unique()

<StringArray>
['75C364B0', 'D89A6000', '53072532', '99693AF2', '075F66DC', 'AC300778',
 'AE908C0E', 'DA37D104', '97171AAE', '1AE72FA6',
 ...
 'A2EF10A1', '9C834865', '4855E456', '5A59BF96', '95B3F7C5', '393732B9',
 '726ED10E', '4F4823F1', 'D47F5181', '66FADEDF']
Length: 368, dtype: str

We are working with 368 unique SKUS.

### How many unique Manufacturing Sites are we working with?

In [ ]:
transaction_FG_MTS["manufacturing_site"].drop_duplicates().count()

np.int64(20)

In [ ]:
transaction_FG_MTS["manufacturing_site"].unique()

<StringArray>
[  'JP-41',   'US-B4',   'CA-66',   'JP-E2', 'missing',   'JP-8F',   'US-51',
   'US-CC',   'JP-B4',   'US-F3',   'CA-B0',   'US-46',   'US-95',   'PL-E4',
   'US-E8',   'CA-DC',   'PL-9D',   'CA-E2',   'CA-2B',   'US-39']
Length: 20, dtype: str

We are working with 20 unique manufacturing sites, putting those with an empty value in their own category. So that would likely actually be 19 different manufacturing sites. 

### How many Divisions are we working with?

In [ ]:
transaction_FG_MTS["division_code"].drop_duplicates().count()

np.int64(55)

In [ ]:
transaction_FG_MTS["division_code"].unique()

<StringArray>
[   '23F0',    '219B',    '81FE',    'B21F',    '15DF',    'F996',    'A57B',
    '0377',    '6E75',    '6661',    'D66F',    '3E44',    '58D1', 'missing',
    '8399',    'BBEF',    '5E37',    '0F54',    'ACA9',    '77CA',    '9B75',
    '2807',    'D4C8',    'D0AA',    'A33E',    '6AF1',    '40B7',    '9F1F',
    'B298',    '7A77',    '2D12',    '7454',    'BC95',    'A8D6',    '17B8',
    '0BC0',    '98BC',    '66DE',    '5873',    '2452',    'ED17',    'F072',
    'D81C',    '69FC',    'B021',    '21F9',    '4AE0',    '0FEE',    'FD41',
    'D105',    'BBC9',    '5A0D',    '356B',    'B813',    '641C']
Length: 55, dtype: str

We are working with 55 unique divisions, putting those with an empty value in their own category. Similar to above, we only know of 54 division codes. 

### What are the top and bottom 10 transactions by order quantity?

Top 10:

In [ ]:
transaction_FG_MTS.sort_values("order_quantity", ascending = False).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
754319,BB9C0690,FG,MTS,28,1008.318397,CA-B0,0377,2025-11-18,187,Canada
580937,CF2CC8C3,FG,MTS,28,1002.957757,US-B4,B21F,2025-03-22,185,United States
275864,DB131462,FG,MTS,28,1008.340512,CA-66,9B75,2024-01-21,184,Canada
633945,3F4A5F06,FG,MTS,28,995.386175,JP-41,15DF,2025-06-04,184,Japan
464404,033E32E7,FG,MTS,28,1028.834974,missing,ACA9,2024-10-11,183,missing
251602,6045B663,FG,MTS,28,1003.757536,CA-66,ACA9,2023-12-18,182,Canada
411599,8126FFFD,FG,MTS,28,994.687522,US-B4,15DF,2024-07-28,181,United States
390349,6AC708C5,FG,MTS,28,1003.846519,missing,6E75,2024-06-29,180,missing
313649,5FFD6CCA,FG,MTS,28,1002.300571,missing,5E37,2024-03-13,180,missing
520992,DC202113,FG,MTS,28,1005.729106,missing,81FE,2024-12-28,180,missing


Bottom 10:

In [ ]:
transaction_FG_MTS.sort_values("order_quantity").head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
356682,DA2B4350,FG,MTS,14,46.420439,PL-E4,8399,2024-05-13,0,Poland
356721,6622FAFA,FG,MTS,21,188.067289,US-B4,B21F,2024-05-13,0,United States
534856,1DDD171D,FG,MTS,14,49.870525,missing,3E44,2025-01-17,0,missing
31979,C347FD10,FG,MTS,28,109.915381,CA-66,219B,2023-02-14,0,Canada
111064,D96E5674,FG,MTS,14,49.662555,US-39,missing,2023-06-05,0,United States
219623,EC9094D3,FG,MTS,21,197.187788,missing,81FE,2023-11-03,0,missing
711736,733E5368,FG,MTS,14,50.952380,JP-41,5E37,2025-09-20,0,Japan
402349,F37AC76A,FG,MTS,14,57.094573,PL-E4,B21F,2024-07-15,0,Poland
148172,B8E6D50A,FG,MTS,28,102.203520,US-F3,0377,2023-07-26,0,United States
673876,08E3F51E,FG,MTS,14,55.700767,CA-66,ED17,2025-07-29,0,Canada


### What are the top and bottom 10 transactions by total sales value?

We added a `sales_value` column after `order_quantity` to easily sort by these values.

In [ ]:
transaction_FG_MTS.insert(9, "sales_value", transaction["order_quantity"] * transaction["unit_price"])

Top 10:

In [ ]:
transaction_FG_MTS.sort_values("sales_value", ascending = False).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,sales_value,country
754319,BB9C0690,FG,MTS,28,1008.318397,CA-B0,0377,2025-11-18,187,188555.540274,Canada
464404,033E32E7,FG,MTS,28,1028.834974,missing,ACA9,2024-10-11,183,188276.800322,missing
580937,CF2CC8C3,FG,MTS,28,1002.957757,US-B4,B21F,2025-03-22,185,185547.185076,United States
275864,DB131462,FG,MTS,28,1008.340512,CA-66,9B75,2024-01-21,184,185534.654155,Canada
633945,3F4A5F06,FG,MTS,28,995.386175,JP-41,15DF,2025-06-04,184,183151.056261,Japan
251602,6045B663,FG,MTS,28,1003.757536,CA-66,ACA9,2023-12-18,182,182683.871515,Canada
520992,DC202113,FG,MTS,28,1005.729106,missing,81FE,2024-12-28,180,181031.239026,missing
390349,6AC708C5,FG,MTS,28,1003.846519,missing,6E75,2024-06-29,180,180692.373340,missing
313649,5FFD6CCA,FG,MTS,28,1002.300571,missing,5E37,2024-03-13,180,180414.102792,missing
187693,AE908C0E,FG,MTS,28,1013.553887,US-B4,6E75,2023-09-19,178,180412.591878,United States


Bottom 10:

In [ ]:
transaction_FG_MTS.sort_values("sales_value").head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,sales_value,country
289,59DA300E,FG,MTS,14,45.564111,CA-66,6E75,2023-01-01,0,0.0,Canada
177774,4F93CAC8,FG,MTS,14,50.121895,missing,58D1,2023-09-05,0,0.0,missing
25339,1C17E3B5,FG,MTS,14,52.227066,JP-41,219B,2023-02-05,0,0.0,Japan
161487,EAD56D2D,FG,MTS,14,57.517556,CA-66,5E37,2023-08-14,0,0.0,Canada
755787,32C80C5D,FG,MTS,28,99.988209,JP-41,6E75,2025-11-21,0,0.0,Japan
355587,97E3DC2E,FG,MTS,14,52.446068,US-B4,2D12,2024-05-11,0,0.0,United States
73371,0EA738D6,FG,MTS,14,53.985355,US-B4,8399,2023-04-13,0,0.0,United States
740800,B13BB4F3,FG,MTS,21,219.934190,CA-66,0F54,2025-10-31,0,0.0,Canada
224596,030D6C59,FG,MTS,14,47.201922,CA-66,23F0,2023-11-10,0,0.0,Canada
610201,5F8DF18E,FG,MTS,14,50.478969,JP-41,B21F,2025-05-02,0,0.0,Japan


# Assignment 2

In [ ]:
# setup
import pandas as pd
from scipy import stats
from scipy.stats.mstats import mquantiles

### Applying our changes from Assignment 1:

In [ ]:
# data
transaction = pd.read_csv("../data/data.zip")
transaction

#dataframe format
transaction.columns = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price", "manufacturing_site", "division_code", "transaction_date", "order_quantity"]   #from 2_data_cleaning

#transaction data type fix
transaction["transaction_date"] = pd.to_datetime(transaction["transaction_date"], format='%Y-%m-%d') # from 2_data_cleaning
transaction.info()

# fix lead time
transaction[transaction['lead_time'] == 2800]
transaction.loc[
    transaction['lead_time'] == 2800,
    'lead_time'] = 28

#unit price and order quantity values fix
transaction['unit_price'] = transaction['unit_price'].abs()
transaction['order_quantity'] = transaction['order_quantity'].abs()

#Na values
Transaction_ValuesMissing = ['sku_number','inventory_type', 'stocking_type', 'manufacturing_site', 'division_code'] 
transaction[Transaction_ValuesMissing] = transaction[Transaction_ValuesMissing].fillna("missing")
transaction


FileNotFoundError: [Errno 2] No such file or directory: '../data/transaction.zip'

The changes that were mentioned in Assignment 1 have been made through this shell above. We now don't have any data loss as we have changed NA values into "missing". We are now ready to start the safety stock calculation process.

## 1. Required Transformation

In [ ]:
mts = transaction.query('stocking_type in "MTS"')
mts

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
1,83992E02,missing,MTS,21,202.017834,CA-66,ACA9,2023-01-01,39
2,75C364B0,FG,MTS,28,986.009314,JP-41,23F0,2023-01-01,111
3,2C9772C9,missing,MTS,28,1007.216590,CA-66,9B75,2023-01-01,89
4,ACCCE163,WIP,MTS,28,986.998503,CA-66,6E75,2023-01-01,80
14,D89A6000,FG,MTS,28,988.743492,US-B4,219B,2023-01-01,75
...,...,...,...,...,...,...,...,...,...
784634,F8B91C1B,WIP,MTS,28,998.021118,missing,23F0,2025-12-30,111
784635,6664F459,FG,MTS,28,99.200335,CA-66,D66F,2025-12-30,1
784636,3DE1D314,FG,MTS,28,1008.523300,missing,81FE,2025-12-30,115
784637,DB131462,FG,MTS,28,1008.340512,CA-2B,6AF1,2025-12-30,122


Items with the stocking type `MTS` were filtered and stored in a separate dataframe.

In [ ]:
#Create new data frame grouped by sku and aggregated quantity and lead time for SS calculation
mts_df =(mts
         .groupby("sku_number")
         .agg(min_qty=("order_quantity", "min"),
              max_qty=("order_quantity", "max"),
              avg_qty=("order_quantity", "mean"),
              median_qty=("order_quantity", "median"),
              variance_qty=("order_quantity", "var"),
              std_qty=("order_quantity", "std"),
              avg_lead_time=("lead_time", "mean"))
              .reset_index())
mts_df

,sku_number,min_qty,max_qty,avg_qty,median_qty,variance_qty,std_qty,avg_lead_time
0,00234778,0,36,9.117647,7.0,58.679091,7.660228,14.000000
1,01517B90,24,169,100.552604,100.0,420.922537,20.516397,28.000000
2,020287B4,0,77,24.360577,22.0,273.990153,16.552648,21.000000
3,02144443,13,167,99.677966,101.0,409.478645,20.235579,28.000000
4,0273F5A5,0,86,23.654639,21.5,281.229255,16.769891,21.000000
...,...,...,...,...,...,...,...,...
504,FD816E1D,0,76,20.574737,17.0,254.785014,15.961987,19.526316
505,FE482594,0,83,23.474801,21.0,282.936198,16.820707,21.000000
506,FE564816,0,38,10.760417,9.0,68.172666,8.256674,28.000000
507,FEAD0FF0,0,169,87.996196,95.0,1234.273059,35.132223,26.315739


**Explanation:**

Creating this table provides us the calculations we need to prepare for our safety stock calculations. This table includes insightful statistic features to see the minimum, maximum, average, median, and variance quantities for each SKU. 

## 2. Safety Stock Calculation

In [ ]:
#Set the z-score for each service level, under a normal distribution
mts_z75 = stats.norm.ppf(.75)
mts_z90 = stats.norm.ppf(.90)
mts_z95 = stats.norm.ppf(.95)

print(mts_z75, mts_z90, mts_z95)

0.6744897501960817 1.2815515655446004 1.644853626951472


This gives us ths the z-scores for each service level. It is 0.674 for 75%, 1.282 for 90%, and 1.645 for 95%. We will use these values for the saftey stock calculation. These will always be the same, no matter what data we are working with. 

In [ ]:
#Add column for safety stock at 75% service level
mts_df["safety_stock_75"] = (
    mts_z75
    *mts_df["std_qty"]
    *mts_df["avg_lead_time"]** 0.5      #The "** 0.5" calculates the square root by applying the power of 0.5 to the value. (same as square root)
)
mts_df

,sku_number,min_qty,max_qty,avg_qty,median_qty,variance_qty,std_qty,avg_lead_time,safety_stock_75
0,00234778,0,36,9.117647,7.0,58.679091,7.660228,14.000000,19.332190
1,01517B90,24,169,100.552604,100.0,420.922537,20.516397,28.000000,73.224339
2,020287B4,0,77,24.360577,22.0,273.990153,16.552648,21.000000,51.162585
3,02144443,13,167,99.677966,101.0,409.478645,20.235579,28.000000,72.222081
4,0273F5A5,0,86,23.654639,21.5,281.229255,16.769891,21.000000,51.834063
...,...,...,...,...,...,...,...,...,...
504,FD816E1D,0,76,20.574737,17.0,254.785014,15.961987,19.526316,47.574305
505,FE482594,0,83,23.474801,21.0,282.936198,16.820707,21.000000,51.991130
506,FE564816,0,38,10.760417,9.0,68.172666,8.256674,28.000000,29.468600
507,FEAD0FF0,0,169,87.996196,95.0,1234.273059,35.132223,26.315739,121.559462


In [ ]:
#Add column for safety stock at 90% service level
mts_df["safety_stock_90"] = (
    mts_z90
    *mts_df["std_qty"]
    *mts_df["avg_lead_time"]** 0.5 
)
mts_df

,sku_number,min_qty,max_qty,avg_qty,median_qty,variance_qty,std_qty,avg_lead_time,safety_stock_75,safety_stock_90
0,00234778,0,36,9.117647,7.0,58.679091,7.660228,14.000000,19.332190,36.731765
1,01517B90,24,169,100.552604,100.0,420.922537,20.516397,28.000000,73.224339,139.128528
2,020287B4,0,77,24.360577,22.0,273.990153,16.552648,21.000000,51.162585,97.210507
3,02144443,13,167,99.677966,101.0,409.478645,20.235579,28.000000,72.222081,137.224207
4,0273F5A5,0,86,23.654639,21.5,281.229255,16.769891,21.000000,51.834063,98.486336
...,...,...,...,...,...,...,...,...,...,...
504,FD816E1D,0,76,20.574737,17.0,254.785014,15.961987,19.526316,47.574305,90.392663
505,FE482594,0,83,23.474801,21.0,282.936198,16.820707,21.000000,51.991130,98.784769
506,FE564816,0,38,10.760417,9.0,68.172666,8.256674,28.000000,29.468600,55.991260
507,FEAD0FF0,0,169,87.996196,95.0,1234.273059,35.132223,26.315739,121.559462,230.966769


In [ ]:
#Add column for safety stock at 95% service level
mts_df["safety_stock_95"] = (
    mts_z95
    *mts_df["std_qty"]
    *mts_df["avg_lead_time"]** 0.5 
)
mts_df

,sku_number,min_qty,max_qty,avg_qty,median_qty,variance_qty,std_qty,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95
0,00234778,0,36,9.117647,7.0,58.679091,7.660228,14.000000,19.332190,36.731765,47.144710
1,01517B90,24,169,100.552604,100.0,420.922537,20.516397,28.000000,73.224339,139.128528,178.569533
2,020287B4,0,77,24.360577,22.0,273.990153,16.552648,21.000000,51.162585,97.210507,124.768336
3,02144443,13,167,99.677966,101.0,409.478645,20.235579,28.000000,72.222081,137.224207,176.125363
4,0273F5A5,0,86,23.654639,21.5,281.229255,16.769891,21.000000,51.834063,98.486336,126.405844
...,...,...,...,...,...,...,...,...,...,...,...
504,FD816E1D,0,76,20.574737,17.0,254.785014,15.961987,19.526316,47.574305,90.392663,116.017727
505,FE482594,0,83,23.474801,21.0,282.936198,16.820707,21.000000,51.991130,98.784769,126.788879
506,FE564816,0,38,10.760417,9.0,68.172666,8.256674,28.000000,29.468600,55.991260,71.864004
507,FEAD0FF0,0,169,87.996196,95.0,1234.273059,35.132223,26.315739,121.559462,230.966769,296.442639


**Explanation:**

The code above has added the safety stock under three different confidence levels consisting of 75%, 90%, and 95%. This allows us to see different levels of safety stock we would need to keep into consideration for the make to stock (`MTS`) SKU items. 

## 3. Safety Stock Distribution

In [ ]:
#95% safety stock
#Sort for largest
mts_df.sort_values(by="safety_stock_95", ascending=False).head(1)

,sku_number,min_qty,max_qty,avg_qty,median_qty,variance_qty,std_qty,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95
484,F39BBC11,0,161,77.029494,91.0,1900.860374,43.598857,26.487844,151.346982,287.563988,369.084382


At a 95% service level SKU `F39BBC11` has the largest safety stock, with a quantity of 369.084382, which we will round to 370 units. 

In [ ]:
#Sort for smallest
mts_df.sort_values(by="safety_stock_95").head(1)

,sku_number,min_qty,max_qty,avg_qty,median_qty,variance_qty,std_qty,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95
90,2462CBBC,0,22,7.840426,7.0,24.84523,4.984499,14.0,12.579428,23.901305,30.676993


At a 95% service level SKU `2462CBBC` has the smallest safety stock, with a quantity of 30.676993, which we will round to 31 units. 

In [ ]:
#Mean of safety stock column across all SKUs
mts_df["safety_stock_95"].mean()

np.float64(124.1958689900739)

The average safety stock across all SKU's is 124.2 units at a 95% service level. 

## Side Quest

We grouped `MTS` items by `SKU` and `transaction_date` and found total order quantities, and set it to `daily_order_qty`.

In [ ]:
daily_order_qty = (
    mts.groupby(["sku_number", "transaction_date"], as_index=False)
    ["order_quantity"].sum()
)
daily_order_qty

,sku_number,transaction_date,order_quantity
0,00234778,2023-01-21,31
1,00234778,2023-01-28,7
2,00234778,2023-02-23,7
3,00234778,2023-02-24,5
4,00234778,2023-03-01,3
...,...,...,...
227198,FF6FAC01,2025-12-25,267
227199,FF6FAC01,2025-12-26,182
227200,FF6FAC01,2025-12-27,85
227201,FF6FAC01,2025-12-29,239


 Then we used `mquantiles()` from SciPy to find the 0.75, 0.90, and 0.95 percentiles for each SKU. We learned that empirical quantiles actually use the historical demand instead of just assuming it follows the normal distribution bell shaped curve.  

In [ ]:
quantiles = pd.DataFrame(columns = ["sku_number", "safety_stock_75", "safety_stock_90", "safety_stock_95"])
orders = daily_order_qty[["sku_number", "order_quantity"]]
for sku in orders["sku_number"].unique():
	prob = mquantiles(orders.query('sku_number == @sku')["order_quantity"], prob = [0.75, 0.9, 0.95], alphap = 0, betap = 1)
	quantiles.loc[len(quantiles)] = [sku, prob[0], prob[1], prob[2]]
quantiles.sort_values("safety_stock_95", ascending = False)

,sku_number,safety_stock_75,safety_stock_90,safety_stock_95
5,030D6C59,656.00,824.0,927.00
252,7B294AD6,651.00,803.7,909.80
184,58C39130,467.00,597.7,693.35
283,91F6F23F,455.75,604.6,692.90
494,F8B91C1B,471.00,618.0,692.40
...,...,...,...,...
141,43C96AB6,14.00,18.0,18.50
127,3A5E9C6B,11.00,17.0,18.00
175,53F0A631,11.25,15.3,17.65
211,61D81899,12.50,16.0,17.50


# References:
- Bobbitt, Z. (2021, April 26). Percentile vs. quartile vs. quantile: What’s the difference? Statology. https://www.statology.org/percentile-vs-quartile-vs-quantile/

We used this to understand quantiles and how we can relate it to safety stock calculation. 

- Stack Overflow(2022). "How do I calculate square root in Python?". Retrieved November 17, 2026. https://stackoverflow.com/questions/70793490/how-do-i-calculate-square-root-in-python 

When searching for "How to square root in python" on Stack Overflow, we come across a post (Stack Overflow, 2022) that has last been edited this year on August 24th that indicates the snippet below:

```
Option 2: Fractional exponent
The power operator (**) or the built-in pow() function can also be used to calculate a square root. Mathematically speaking, the square root of a equals a to the power of 1/2.

The power operator requires numeric types and matches the conversion rules for binary arithmetic operators, so in this case it will return either a float or a complex number

>>> 9 ** (1/2)
3.0
>>> 9 ** .5  # Same thing
3.0
>>> 2 ** .5
1.4142135623730951

```